# Syllabus Time-Candidate Classifier — Training

Fine-tunes a multilingual encoder to classify each date/time candidate so only real
class meeting times reach the calendar. **Priority: `class_schedule` precision** — it is
far worse to add a wrong class than to miss one.

**Before running:** *Runtime → Change runtime type → GPU (T4 is enough).*

## 0. Clone the repo and install

In [ ]:
!git clone https://github.com/hyunnow/sllallm.git
%cd sllallm
!pip -q install -e ".[train]"

## 1. Upload the data splits

The labeled dataset lives **outside git** (it contains real syllabus text). Upload the three
split files from your local `data/splits/` — `train.jsonl`, `val.jsonl`, `test.jsonl`.

> Prefer Google Drive for repeat runs (see the commented alternative below).

In [ ]:
import os, pathlib
pathlib.Path('data/splits').mkdir(parents=True, exist_ok=True)

from google.colab import files
print('Select train.jsonl, val.jsonl, test.jsonl from your local data/splits/ …')
uploaded = files.upload()
for name, content in uploaded.items():
    open(f'data/splits/{name}', 'wb').write(content)
print('splits present:', sorted(p for p in os.listdir('data/splits') if p.endswith('.jsonl')))

# --- Google Drive alternative (uncomment) ---
# from google.colab import drive; drive.mount('/content/drive')
# !cp /content/drive/MyDrive/sllallm_splits/*.jsonl data/splits/

## 2. Augment the TRAIN split (val/test stay untouched)

Surface noise (OCR confusion, synonym/format swaps) with labels preserved — teaches
robustness without leaking. Regenerated here so you only upload the 3 real splits.

In [ ]:
!python scripts/05_augment.py

## 3. Sanity check

In [ ]:
import json, collections, os
for s in ['train', 'val', 'test', 'train_aug']:
    p = f'data/splits/{s}.jsonl'
    if os.path.exists(p):
        rows = [json.loads(l) for l in open(p)]
        d = collections.Counter(r['label'] for r in rows)
        print(f'{s:10} {len(rows):6}  class_schedule={d["class_schedule"]:5}  office={d["instructor_office_hours"]+d["ta_office_hours"]:5}')

## 4. Train

Class-weighted / focal loss for imbalance, a conservative `class_schedule` confidence
threshold (demote to runner-up when unsure), and evaluation on the **real holdout** with
the risk-aware metrics. Start with `klue/roberta-base`.

In [ ]:
import sys; sys.path.insert(0, 'src')
from syllabus_classifier.common.seed import set_seed
from syllabus_classifier.model.train import train

set_seed(42)
metrics_klue = train(train_file='data/splits/train_aug.jsonl', out_dir='checkpoints/klue')

## 5. Read the holdout report

The number that matters most is **class_schedule precision** (target 0.98+) and
**office→class rate** (target ~0).

In [ ]:
import json
m = json.load(open('checkpoints/klue/holdout_metrics.json'))
print(f"class_schedule precision : {m['class_schedule_precision']:.3f}")
print(f"class_schedule recall    : {m['class_schedule_recall']:.3f}")
print(f"office -> class rate     : {m['office_hours_to_class_schedule_rate']:.3f}")
print(f"macro F1                 : {m['macro_f1']:.3f}")

# confusion matrix (rows = true, cols = pred), office<->class corner highlighted
cm = m['confusion_matrix']; labels = list(cm.keys())
short = {l: l[:10] for l in labels}
print('\n' + ' ' * 12 + ' '.join(f'{short[l]:>11}' for l in labels))
for t in labels:
    print(f'{short[t]:>11} ' + ' '.join(f'{cm[t][p]:>11}' for p in labels))

## 6. (Optional) Compare with xlm-roberta-base

`xlm-roberta-base` is stronger on mixed Korean/English text. Swap the encoder and retrain.

In [ ]:
set_seed(42)
metrics_xlmr = train(encoder='xlm-roberta-base',
                     train_file='data/splits/train_aug.jsonl', out_dir='checkpoints/xlmr')
print('klue class_prec:', round(metrics_klue['class_schedule_precision'], 3),
      '| xlmr class_prec:', round(metrics_xlmr['class_schedule_precision'], 3))

## 7. Save the model to Drive

In [ ]:
from google.colab import drive; drive.mount('/content/drive')
!mkdir -p '/content/drive/MyDrive/sllallm_model'
!cp -r checkpoints/klue/best '/content/drive/MyDrive/sllallm_model/klue_best'
print('saved to Drive: sllallm_model/klue_best')

---
**Next:** tune the `class_schedule_confidence_threshold` in `config/train.yaml` to trade recall
for precision, then plug the trained model behind the same `Classifier.predict` interface as
`HeuristicClassifier` so the rest of the pipeline (validator, KB resolver) is unchanged.